# LABEL0 P_w0 — 5분 점수의 끝 순간 항 무게를 0으로 바꾼 학생 (무리 도착 세계)

5분 점수 = 5분 지연 + 끝 순간 멈춘 차 × 무게입니다. 지금 무게 75 대신 0으로 라벨을 다시 계산해 학생을 다시 학습하고, 한 시간 SUMO로 잽니다.
자료·신경망·학습 설정은 PT1와 같고 새 굴림은 없습니다. 방법과 읽는 법은 `results/explore/label0_20261102/사전등록.md`에 고정했습니다.

노트북은 후보 4개(P_w0·P_w25·R_w0·R_w25)와 판정 1개입니다. **후보마다 런타임을 하나씩 따로** 씁니다. 순서는 상관없습니다.

1. 런타임 유형 **CPU + 높은 RAM(CPU 8개)** 을 고르고 **런타임 → 모두 실행**을 누르세요. Drive 접근을 허용하세요.
2. 순서: 묶음 검증 → 패키지 → 시험 → 학생 학습(약 5분) → 작은 실행(약 10분) → 본 평가(252판, CPU 8개 기준 약 2~3시간, 어림).
3. 본 평가는 백그라운드로 돕니다. 끊기면 다시 **모두 실행**하세요. 끝난 판은 건너뜁니다.
4. 후보 4개가 끝나면 `LABEL0_judge`를 실행하고 "끝났어"라고 알려 주세요. 결과: Drive `FutureTraffic/explore_label0/`.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json

drive.mount("/content/drive")
VARIANT = "P_w0"
DRIVE = Path("/content/drive/MyDrive/FutureTraffic")
ARCHIVE = DRIVE / "FutureTraffic_LABEL0.zip"
DATA = DRIVE / "explore_platoon1/data"                                # 기존 자료(읽기만)
OLD_STUDENTS = DRIVE / "explore_platoon1/students"                    # 무게 75 학생(읽기만)
OLD_EVAL = DRIVE / "explore_platoon1/eval/PT1"                      # 무게 75 평가(읽기만)
PLATOON0 = DRIVE / "explore_platoon0"                  # 평가 지문 대조(읽기만)
OUT = DRIVE / "explore_label0" / VARIANT
WORKSPACE = Path("/content/FutureTraffic_LABEL0")
MIN_CPUS = 8
for path in (ARCHIVE, DATA / "experiment.json", OLD_STUDENTS / "seed_42" / "model.pt", OLD_EVAL / "models.json", PLATOON0 / "experiment.json"):
    assert path.exists(), f"필요한 파일이 없습니다: {path}"
CPUS = os.cpu_count() or 1
print("CPU 개수:", CPUS, "| 후보:", VARIANT, "| 결과 폴더:", OUT)
assert CPUS >= MIN_CPUS, (f"CPU가 {CPUS}개입니다. 런타임 → 런타임 유형 변경에서 CPU와 '높은 RAM'을 고른 뒤 다시 실행하세요.")

In [ ]:
# ZIP 전체의 파일 목록·SHA-256을 확인한 뒤 검증된 전용 추출기를 사용합니다.
import hashlib, stat, types, zipfile
from pathlib import PurePosixPath

with zipfile.ZipFile(ARCHIVE) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)) and names.count("manifest.json") == 1, "ZIP 파일 목록 오류"
    package_manifest = json.loads(archive.read("manifest.json"))
    assert package_manifest.get("kind") == "WEIGHT1_PACKAGE_V1", "검증할 수 없는 ZIP 형식입니다"
    assert package_manifest.get("experiment") == "LABEL0_V1", "LABEL0 전용 ZIP이 아닙니다"
    assert package_manifest.get("source_models_included") is False, "모델·결과가 든 묶음은 쓰지 않습니다"
    expected = package_manifest["files"]
    assert expected and "manifest.json" not in expected and set(names) == set(expected) | {"manifest.json"}, "ZIP과 manifest 목록 불일치"
    for info in archive.infolist():
        name, path = info.filename, PurePosixPath(info.filename)
        assert path.parts and not path.is_absolute() and ".." not in path.parts and path.as_posix() == name, "잘못된 ZIP 경로"
        assert "\\" not in name and not info.is_dir() and not stat.S_ISLNK(info.external_attr >> 16), "잘못된 ZIP 파일 형식"
        if name != "manifest.json":
            assert hashlib.sha256(archive.read(name)).hexdigest() == expected[name], f"SHA-256 불일치: {name}"
    source_code = archive.read("src/explore_weight_package.py").decode("utf-8")

bootstrap = types.ModuleType("label0_verified_package")
bootstrap.__file__ = str(WORKSPACE / "src/explore_weight_package.py")
exec(compile(source_code, bootstrap.__file__, "exec"), bootstrap.__dict__)
package_manifest = bootstrap.extract_verified(ARCHIVE, WORKSPACE)
print(f"검증·추출 완료: {len(package_manifest['files'])}개 파일 → {WORKSPACE} | 잠금 {package_manifest['lock_sha256'][:12]} "
      f"| PLATOON0 설정 {package_manifest['platoon0_config_sha256'][:12]} | 측정 계획 {package_manifest['plan_sha256'][:12]}")

In [ ]:
import subprocess, importlib.metadata

os.environ["FOCUSED_RULESET"] = "pet1"
os.environ["FOCUSED_NORTH_EXT"] = "240"
# ARENA3 심판 기준선과 같은 평가 지문이 되도록 같은 패키지 버전을 씁니다.
EXPECTED = {"numpy": "2.1.3", "torch": "2.11.0+cpu", "stable-baselines3": "2.9.0",
            "sumo-rl": "1.4.5", "eclipse-sumo": "1.27.1"}

def installed_versions():
    versions = {}
    for name in EXPECTED:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions

before = installed_versions()
if before == EXPECTED:
    print("이전 실험과 패키지 버전이 같아 재설치를 건너뜁니다.")
else:
    print("이전 실험과 같은 패키지 버전을 준비합니다:", before)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3", "stable-baselines3==2.9.0", "eclipse-sumo==1.27.1"],
                   cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.11.0+cpu", "--index-url", "https://download.pytorch.org/whl/cpu"],
                   cwd=WORKSPACE, check=True)
after = installed_versions()
assert after == EXPECTED, f"패키지 버전이 이전 실험과 다릅니다(평가 지문이 달라져 기준선과 비교할 수 없어 멈춥니다): {after}"
print("실행 패키지 검증 통과:", after)

In [ ]:
sys.path.insert(0, str(WORKSPACE / "src"))
import arena3_colab as colab
from IPython.display import display, Markdown

colab.run_live(["-m", "unittest", "discover", "-s", "tests", "-p", "test_explore_label0.py"], WORKSPACE)
print("실행 전 검사(SUMO 없이) 통과")

In [ ]:
# 1) 학생 학습(약 5분). 이미 있으면 다시 쓰지 않습니다.
colab.run_live(["src/explore_label0.py", "train", "--variant", VARIANT, "--data", str(DATA), "--out", str(OUT / "students")], WORKSPACE)
print("학생 학습 끝")

In [ ]:
# 2) 작은 실행(약 10분): 무게 75 라벨이 원래 자료와 같은지, 후보 라벨이 달라졌는지, 학생 한 판이 온전한지 봅니다.
SMOKE = OUT / "smoke"
previous = colab.read_json(SMOKE / "smoke.json")
if previous and previous.get("ok"):
    print("이전 작은 실행이 통과해 건너뜁니다.")
else:
    colab.run_live(["src/explore_label0.py", "smoke", "--variant", VARIANT, "--data", str(DATA), "--students", str(OUT / "students"),
                    "--reference-students", str(OLD_STUDENTS), "--reference-eval", str(OLD_EVAL), "--out", str(SMOKE),
                    "--work", "/content/l0_smoke", "--evaluation", str(PLATOON0)], WORKSPACE, check=False)
s = colab.read_json(SMOKE / "smoke.json")
assert s and s.get("ok"), f"작은 실행 점검 실패 — 이 칸의 출력을 저에게 보내 주세요: {s and s.get('checks')}"
print("작은 실행 통과:", s["checks"], "| 지연", s["delay_s"], "(무게 75:", s["delay_w75_s"], ") | 무게 75 재학습 일치:", s["retrain_w75_same_model"])

In [ ]:
# 3) 본 평가(모델 3개 × 84상황 = 252판). 백그라운드로 돌아 셀을 멈춰도 계속 돕니다.
#    런타임이 끊기면 처음 네 칸 → 이 칸을 다시 실행하세요. 끝난 판은 건너뜁니다.
assert "colab" in globals(), "처음 네 칸(1~4)을 먼저 실행하세요. 런타임이 끊기면 변수와 작업 폴더가 사라집니다."
LOGS = OUT / "logs"
EVAL = OUT / "eval"

def describe(name):
    done = len(list((EVAL / "runs").glob("*/*/cache.json")))
    failed = len(list((EVAL / "runs").glob("*/*.failed.json")))
    return f"끝난 판 {done}/252, 실패 기록 {failed}"

pids = colab.launch({f"label0_{VARIANT}": ["src/explore_label0.py", "evaluate", "--variant", VARIANT, "--students", str(OUT / "students"),
                                           "--out", str(EVAL), "--work", "/content/l0", "--workers", str(CPUS)]}, WORKSPACE, LOGS)
code = colab.watch(pids, LOGS, 300, describe)
assert code == 0, "본 평가에서 오류가 났습니다. 위 로그를 저에게 보내 주세요(다시 실행하면 끝난 판은 건너뜁니다)."
print("본 평가 끝. 후보 4개가 끝나면 LABEL0_judge를 실행하세요.")